# Process the different formats provided by different data sets separately

In [ ]:
# R
library(Signac)
library(Seurat)

setwd('peak_gene/')

In [ ]:
# 10x datasets
list = paste0('10x_datasets_', 1:10)

rna <- Read10X(paste0(list[1], '/raw/filtered_feature_bc_matrix/'))[["Gene Expression"]]
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/', list[1], '.rds'))
atac <- atac[,colnames(rna)]
colnames(rna) <- paste0(paste0(list[1],'-',
                               colnames(rna)))
colnames(atac) <- paste0(paste0(list[1],'-',
                               colnames(atac)))

for (i in 2:10){
  rnaa <- Read10X(paste0(list[i], '/raw/filtered_feature_bc_matrix/'))[["Gene Expression"]]
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/', list[i], '.rds'))
  ataca <- ataca[,colnames(rnaa)]
  colnames(rnaa) <- paste0(paste0(list[i],'-',
                                 colnames(rnaa)))
  colnames(ataca) <- paste0(paste0(list[i],'-',
                                  colnames(ataca)))
  
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}

saveRDS(rna, 'z_others/3_rna_atac_cell/data/10x_datasets-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/10x_datasets-atac.rds')

In [ ]:
# Boukhaled_Nat_Immunol_2022
list = list.files('Boukhaled_Nat_Immunol_2022/processed/')

rna <- Read10X(paste0('Boukhaled_Nat_Immunol_2022/raw/',list[1],'/filtered_feature_bc_matrix/'))[["Gene Expression"]]
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/Boukhaled_Nat_Immunol_2022-', list[1], '.rds'))
atac <- atac[,colnames(rna)]
colnames(rna) <- paste0('Boukhaled_Nat_Immunol_2022-',
                               substr(colnames(rna), 1, 17),
                               list[1])
colnames(atac) <- paste0('Boukhaled_Nat_Immunol_2022-',
                                substr(colnames(atac), 1, 17),
                                list[1])

for (i in 2:10){
  rnaa <- Read10X(paste0('Boukhaled_Nat_Immunol_2022/raw/',list[i],'/filtered_feature_bc_matrix/'))[["Gene Expression"]]
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/Boukhaled_Nat_Immunol_2022-', list[i], '.rds'))
  ataca <- ataca[,colnames(rnaa)]
  colnames(rnaa) <- paste0('Boukhaled_Nat_Immunol_2022-',
                          substr(colnames(rnaa), 1, 17),
                          list[i])
  colnames(ataca) <- paste0('Boukhaled_Nat_Immunol_2022-',
                           substr(colnames(ataca), 1, 17),
                           list[i])
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}

saveRDS(rna, 'z_others/3_rna_atac_cell/data/Boukhaled_Nat_Immunol_2022-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Boukhaled_Nat_Immunol_2022-atac.rds')

In [ ]:
# Chen_Nat_Methods_2022
lane1 <- pbmc[,pbmc$Sample == 'lane1']
lane2 <- pbmc[,pbmc$Sample == 'lane2']

count1 <- lane1@assays[["RNA"]]@counts
count1@Dimnames[[2]] <- paste0('Chen_Nat_Methods_2022-', 
                               substr(count1@Dimnames[[2]], 7, 23),
                               'lane1')
atac1 <- readRDS('z_others/2_cpeak_matrix/data/Chen_Nat_Methods_2022-lane1.rds')
atac1@Dimnames[[2]] <- paste0('Chen_Nat_Methods_2022-', 
                              substr(atac1@Dimnames[[2]], 1, 17),
                              'lane1')
atac1 <- atac1[,colnames(count1)]

count2 <- lane2@assays[["RNA"]]@counts
count2@Dimnames[[2]] <- paste0('Chen_Nat_Methods_2022-', 
                               substr(count2@Dimnames[[2]], 7, 23),
                               'lane2')
atac2 <- readRDS('z_others/2_cpeak_matrix/data/Chen_Nat_Methods_2022-lane2.rds')
atac2@Dimnames[[2]] <- paste0('Chen_Nat_Methods_2022-', 
                              substr(atac2@Dimnames[[2]], 1, 17),
                              'lane2')
atac2 <- atac2[,colnames(count2)]

count <- cbind(count1, count2)
atac <- cbind(atac1, atac2)

saveRDS(count, 'z_others/3_rna_atac_cell/data/Chen_Nat_Methods_2022-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Chen_Nat_Methods_2022-atac.rds')

In [ ]:
# Cheong_Cell_2023_1
remove_last_19_to_1 <- function(x) {
  # Calculate the total length of the string
  str_length <- nchar(x)
  
  # If the string length is less than 19 characters, return the original string
  if (str_length < 19) {
    return(x)
  }
  
  # Extract the substring from the beginning up to the character before the last 19
  # (Effectively removes the last 19 characters)
  return(substring(x, 1, str_length - 19))
}

extract_last_18_to_1 <- function(x) {
  # Calculate the total length of the string
  str_length <- nchar(x)
  
  # If the string length is less than 18 characters, return an empty string
  if (str_length < 18) {
    return("")
  }
  
  # Extract the substring from the 18th-to-last character to the 2nd-to-last character
  return(substring(x, str_length - 17, str_length - 1))
}
result <- as.data.frame(sapply(colnames(GSE196988_RNA_ReadCounts_Final_Multiome), remove_last_19_to_1))
colnames(result) <- 'sample'
result$barcode <- sapply(colnames(GSE196988_RNA_ReadCounts_Final_Multiome), extract_last_18_to_1)

GSE196988_RNA_ReadCounts_Final_Multiome@Dimnames[[2]] <- paste0('Cheong_Cell_2023_1-',
                                                                result$barcode,
                                                                result$sample)
saveRDS(GSE196988_RNA_ReadCounts_Final_Multiome, 'cpeak_matrix/Cheong_Cell_2023_1-rna.rds')

# atac matrix
sample <- list.files('Cheong_Cell_2023_1/cpeak_matrix/sample/')
sam <- sub("\\..*", "", sample[1])
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/Cheong_Cell_2023_1-', sample[1]))
colnames(atac) <- paste0('Cheong_Cell_2023_1-',
                         substr(colnames(atac), 1, 17),
                         sam)
for (i in 2:30){
  sam <- sub("\\..*", "", sample[i])
  ata <- readRDS(paste0('z_others/2_cpeak_matrix/data/Cheong_Cell_2023_1-', sample[i]))
  colnames(ata) <- paste0('Cheong_Cell_2023_1-',
                          substr(colnames(ata), 1, 17),
                          sam)
  atac <- cbind(atac, ata)
}

length(intersect(colnames(GSE196988_RNA_ReadCounts_Final_Multiome),colnames(atac)))
atac <- atac[,colnames(GSE196988_RNA_ReadCounts_Final_Multiome)]
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Cheong_Cell_2023_1-atac.rds')

In [ ]:
# Cheong_Cell_2023_2
count_1 <- Read10X_h5('Cheong_Cell_2023_2/raw/GSM7493830_bmmc_raw_feature_bc_matrix.h5')
count_2 <- Read10X_h5('Cheong_Cell_2023_2/raw/GSM7493831_pbmc_raw_feature_bc_matrix.h5')
meta <- read.csv('Cheong_Cell_2023_2/raw/GSE196988_BM_PBMC_PIE_metadata.csv.gz', row.names = 1)

meta_1 <- meta[meta$dataset == 'bmmcpie',]
meta_2 <- meta[meta$dataset == 'pbmcpie',]

count_1$`Gene Expression` <- count_1$`Gene Expression`[,meta_1$gex_barcode]
count_2$`Gene Expression` <- count_2$`Gene Expression`[,meta_2$gex_barcode]

colnames(count_1$`Gene Expression`) <- paste0('Cheong_Cell_2023_2-',
                                              substr(colnames(count_1$`Gene Expression`),1, 16),
                                              '-bmmcpie')
colnames(count_2$`Gene Expression`) <- paste0('Cheong_Cell_2023_2-',
                                              substr(colnames(count_2$`Gene Expression`),1, 16),
                                              '-pbmcpie')

rna <- cbind(count_1$`Gene Expression`, count_2$`Gene Expression`)

BMMCPIE1 <- readRDS('z_others/2_cpeak_matrix/data/Cheong_Cell_2023_2-BMMCPIE1.rds')
PBMCPIE1 <- readRDS('z_others/2_cpeak_matrix/data/Cheong_Cell_2023_2-PBMCPIE1.rds')
colnames(BMMCPIE1) <- paste0('Cheong_Cell_2023_2-',
                             substr(colnames(BMMCPIE1), 1, 16),
                             '-bmmcpie')
colnames(PBMCPIE1) <- paste0('Cheong_Cell_2023_2-',
                             substr(colnames(PBMCPIE1),1, 16),
                             '-pbmcpie')
atac <- cbind(BMMCPIE1, PBMCPIE1)
atac <- atac[,colnames(rna)]
saveRDS(rna, 'z_others/3_rna_atac_cell/data/Cheong_Cell_2023_2-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Cheong_Cell_2023_2-atac.rds')

In [ ]:
# Hu_BioRxiv_2023
atac <- readRDS('z_others/2_cpeak_matrix/data/Hu_BioRxiv_2023.rds')
atac <- atac[, colnames(pbmc)]
rna <- pbmc@assays[["RNA"]]@counts
colnames(rna) <- paste0('Hu_BioRxiv_2023-', colnames(rna), '-', pbmc@meta.data[["donor"]])
colnames(atac) <- paste0('Hu_BioRxiv_2023-', colnames(atac), '-', pbmc@meta.data[["donor"]])
saveRDS(rna, 'z_others/3_rna_atac_cell/data/Hu_BioRxiv_2023-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Hu_BioRxiv_2023-atac.rds')

In [ ]:
# Lareau_Nat_Genet_2023
meta <- read.table('Lareau_Nat_Genet_2023/raw/Pearson_DOGMA_full_meta_data.tsv', 
                   sep = '\t', 
                   header = T, 
                   row.names = 1)
meta <- meta[colnames(pbmc),]
atac <- readRDS('z_others/2_cpeak_matrix/data/Lareau_Nat_Genet_2023.rds')
atac <- atac[, colnames(pbmc)]
rna <- pbmc@assays[["RNA"]]@counts
colnames(atac) <- paste0('Lareau_Nat_Genet_2023-',substr(colnames(atac),1,17), meta$orig.ident)
colnames(rna) <- paste0('Lareau_Nat_Genet_2023-',substr(colnames(rna),1,17), meta$orig.ident)
saveRDS(rna, 'z_others/3_rna_atac_cell/data/Lareau_Nat_Genet_2023-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Lareau_Nat_Genet_2023-atac.rds')


In [ ]:
# Li_Nat_Biotechnol_2023
meta <- read.csv('Li_Nat_Biotechnol_2023/raw/cell_type_umap.csv', row.names = 1)
meta1 <- meta[meta$batch == '0th day',]
meta2 <- meta[meta$batch == '7th day',]
rownames(meta1) <- substr(rownames(meta1), 1,18)
rownames(meta2) <- substr(rownames(meta2), 1,18)

data1 <- Read10X('Li_Nat_Biotechnol_2023/raw/3423-MV-1/mtx/')
data2 <- Read10X('Li_Nat_Biotechnol_2023/raw/3423-MV-2/mtx/')
MV1 <- readRDS('z_others/2_cpeak_matrix/data/Li_Nat_Biotechnol_2023-MV_1.rds')
MV2 <- readRDS('z_others/2_cpeak_matrix/data/Li_Nat_Biotechnol_2023-MV_2.rds')
rna1 <- data1$`Gene Expression`[,rownames(meta1)]
rna2 <- data2$`Gene Expression`[,rownames(meta2)]
atac1 <- MV1[, rownames(meta1)]
atac2 <- MV2[, rownames(meta2)]
colnames(rna1) <- paste0('Li_Nat_Biotechnol_2023-', substr(colnames(rna1),1,16),'-MV_1')
colnames(atac1) <- paste0('Li_Nat_Biotechnol_2023-', substr(colnames(atac1),1,16),'-MV_1')
colnames(rna2) <- paste0('Li_Nat_Biotechnol_2023-', substr(colnames(rna2),1,16),'-MV_2')
colnames(atac2) <- paste0('Li_Nat_Biotechnol_2023-', substr(colnames(atac2),1,16),'-MV_2')

rna <- cbind(rna1, rna2)
atac <- cbind(atac1, atac2)
saveRDS(rna, 'z_others/3_rna_atac_cell/data/Li_Nat_Biotechnol_2023-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Li_Nat_Biotechnol_2023-atac.rds')

In [ ]:
# Mimitou_Nat_Biotechnol_2021
r1 <- readRDS('Mimitou_Nat_Biotechnol_2021/processed/GSM5065525_LLL_CTRL_.rds')@assays[["RNA"]]@counts
a1 <- readRDS('z_others/2_cpeak_matrix/data/Mimitou_Nat_Biotechnol_2021-LLL_CTRL.rds')[, colnames(r1)]
colnames(r1) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(r1),1,17),
                       'LLL_CTRL')
colnames(a1) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(a1),1,17),
                       'LLL_CTRL')

r2 <- readRDS('Mimitou_Nat_Biotechnol_2021/processed/GSM5065528_LLL_STIM_.rds')@assays[["RNA"]]@counts
a2 <- readRDS('z_others/2_cpeak_matrix/data/Mimitou_Nat_Biotechnol_2021-LLL_STIM.rds')[, colnames(r2)]
colnames(r2) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(r2),1,17),
                       'LLL_STIM')
colnames(a2) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(a2),1,17),
                       'LLL_STIM')

r3 <- readRDS('Mimitou_Nat_Biotechnol_2021/processed/GSM5065531_DIG_CTRL_.rds')@assays[["RNA"]]@counts
a3 <- readRDS('z_others/2_cpeak_matrix/data/Mimitou_Nat_Biotechnol_2021-DIG_CTRL.rds')[, colnames(r3)]
colnames(r3) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(r3),1,17),
                       'DIG_CTRL')
colnames(a3) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(a3),1,17),
                       'DIG_CTRL')

r4 <- readRDS('Mimitou_Nat_Biotechnol_2021/processed/GSM5065534_DIG_STIM_.rds')@assays[["RNA"]]@counts
a4 <- readRDS('z_others/2_cpeak_matrix/data/Mimitou_Nat_Biotechnol_2021-DIG_STIM.rds')[, colnames(r4)]
colnames(r4) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(r4),1,17),
                       'DIG_STIM')
colnames(a4) <- paste0('Mimitou_Nat_Biotechnol_2021-', 
                       substr(colnames(a4),1,17),
                       'DIG_STIM')

rna <- cbind(r1,r2,r3,r4)
atac <- cbind(a1,a2,a3,a4)

saveRDS(rna, 'z_others/3_rna_atac_cell/data/Mimitou_Nat_Biotechnol_2021-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Mimitou_Nat_Biotechnol_2021-atac.rds')


In [ ]:
# NeurIPS_2021
cell <- read.table('NeurIPS_2021/raw/matrix/barcodes.tsv', sep = '\t', header = T)
cell$barcode <- paste0(substr(cell$X, 1, 17), '1')
file <- list.files('NeurIPS_2021/bam/analysis/')

rna <- Read10X(paste0('NeurIPS_2021/bam/analysis/', file[1], '/outs/filtered_feature_bc_matrix/'))$`Gene Expression`
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/NeurIPS_2021-', file[1], '.rds'))
c1 <- cell[cell$Samplename == paste0(file[1], '_multiome'),]
rna <- rna[, c1$barcode]
atac <- atac[, c1$barcode]
colnames(rna) <- paste0('NeurIPS_2021-',
                        substr(colnames(rna),1,17),
                        file[1])
colnames(atac) <- paste0('NeurIPS_2021-',
                         substr(colnames(atac),1,17),
                         file[1])

for (i in 2:13){
  rnaa <- Read10X(paste0('NeurIPS_2021/bam/analysis/', file[i], '/outs/raw_feature_bc_matrix'))$`Gene Expression`
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/NeurIPS_2021-', file[i], '.rds'))
  c1 <- cell[cell$Samplename == paste0(file[i], '_multiome'),]
  rnaa <- rnaa[, c1$barcode]
  ataca <- ataca[, c1$barcode]
  colnames(rnaa) <- paste0('NeurIPS_2021-',
                           substr(colnames(rnaa),1,17),
                           file[i])
  colnames(ataca) <- paste0('NeurIPS_2021-',
                            substr(colnames(ataca),1,17),
                            file[i])
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}

saveRDS(rna, 'z_others/3_rna_atac_cell/data/NeurIPS_2021-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/NeurIPS_2021-atac.rds')

In [ ]:
# Oguchi_Science_2024
meta <- CD4@meta.data
meta$barcode <- substr(rownames(meta), 23, 40)

file <- list.files('Oguchi_Science_2024/raw/E-GEAD-743.matrix/CD4bulk/')
ata <- list.files('z_others/2_cpeak_matrix/data/', pattern = 'Bulk-rCD4')

met <- meta[meta$orig.ident == paste0('Multiome_', file[1]), ]
rna <- Read10X(paste0('Oguchi_Science_2024/raw/E-GEAD-743.matrix/CD4bulk/', file[1]))[["Gene Expression"]][, met$barcode]
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/', ata[1]))[, met$barcode]
colnames(rna) <- paste0('Oguchi_Science_2024-',
                        substr(colnames(rna), 1, 17),
                        file[1])
colnames(atac) <- paste0('Oguchi_Science_2024-',
                         substr(colnames(atac), 1, 17),
                         file[1])

for (i in 2:9) {
  met <- meta[meta$orig.ident == file[i], ]
  met <- meta[meta$orig.ident == paste0('Multiome_', file[i]), ]
  rnaa <- Read10X(paste0('Oguchi_Science_2024/raw/E-GEAD-743.matrix/CD4bulk/', file[i]))[["Gene Expression"]][, met$barcode]
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/', ata[i]))[, met$barcode]
  colnames(rnaa) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(rnaa), 1, 17),
                          file[i])
  colnames(ataca) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(ataca), 1, 17),
                           file[i])
  
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}
CD4bluk_rna <- rna
CD4bluk_atac <- atac

Multiome_D2_Th1or2_1 <- Th1or2[,Th1or2$orig.ident == 'Multiome_D2_Th1or2_1']
Multiome_D2_Th1or2_2 <- Th1or2[,Th1or2$orig.ident == 'Multiome_D2_Th1or2_2']
rna_1 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Th1or2_1/')[["Gene Expression"]][, substr(colnames(Multiome_D2_Th1or2_1), 34, 51)]
rna_2 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Th1or2_2/')[["Gene Expression"]][, substr(colnames(Multiome_D2_Th1or2_2), 34, 51)]
atac_1 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Th1or2_1.rds')[, substr(colnames(Multiome_D2_Th1or2_1), 34, 51)]
atac_2 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Th1or2_2.rds')[, substr(colnames(Multiome_D2_Th1or2_2), 34, 51)]
colnames(rna_1) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(Multiome_D2_Th1or2_1), 34, 50),
                          'D2_Th1or2_1')
colnames(atac_1) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(Multiome_D2_Th1or2_1), 34, 50),
                          'D2_Th1or2_1')
colnames(rna_2) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(Multiome_D2_Th1or2_2), 34, 50),
                          'D2_Th1or2_2')
colnames(atac_2) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(Multiome_D2_Th1or2_2), 34, 50),
                           'D2_Th1or2_2')
Th1or2_rna <- cbind(rna_1, rna_2)
Th1or2_atac <- cbind(atac_1, atac_2)

Activated_CD4_48h <- Activated_CD4_48h[, Activated_CD4_48h$orig.ident == 'Multiome_aCD4_48h']
rna <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D1_Bulk_aCD4_1/')[["Gene Expression"]][, substr(colnames(Activated_CD4_48h), 19, 51)]
atac <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D1_Bulk-aCD4_1.rds')[, substr(colnames(Activated_CD4_48h), 19, 51)]
colnames(rna) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(Activated_CD4_48h), 19, 35),
                          'D1_Bulk_aCD4_1')
colnames(atac) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(Activated_CD4_48h), 19, 35),
                           'D1_Bulk_aCD4_1')
Activated_CD4_48h_rna <- rna
Activated_CD4_48h_atac <- atac

LAG3 <- LAG3[,LAG3$orig.ident == 'Multiome_D2_LAG3_1']
rna <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_LAG3_1/')[["Gene Expression"]][, substr(colnames(LAG3), 20, 37)]
atac <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_LAG3_1.rds')[, substr(colnames(LAG3), 20, 37)]
colnames(rna) <- paste0('Oguchi_Science_2024-',
                        substr(colnames(LAG3), 19, 36),
                        'D2_LAG3_1')
colnames(atac) <- paste0('Oguchi_Science_2024-',
                         substr(colnames(LAG3), 19, 36),
                         'D2_LAG3_1')
LAG3_rna <- rna
LAG3_atac <- atac

D2_Th17_1 <- Th17[,Th17$orig.ident == 'Multiome_D2_Th17_1']
D2_Th17_2 <- Th17[,Th17$orig.ident == 'Multiome_D2_Th17_2']
rna_1 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Th17_1/')[["Gene Expression"]][, substr(colnames(D2_Th17_1), 30, 47)]
rna_2 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Th17_2/')[["Gene Expression"]][, substr(colnames(D2_Th17_2), 30, 47)]
atac_1 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Th17_1.rds')[, substr(colnames(D2_Th17_1), 30, 47)]
atac_2 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Th17_2.rds')[, substr(colnames(D2_Th17_2), 30, 47)]
colnames(rna_1) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(D2_Th17_1), 30, 46),
                          'D2_Th17_1')
colnames(atac_1) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(D2_Th17_1), 30, 46),
                           'D2_Th17_1')
colnames(rna_2) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(D2_Th17_2), 30, 46),
                          'D2_Th17_2')
colnames(atac_2) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(D2_Th17_2), 30, 46),
                           'D2_Th17_2')
Th17_rna <- cbind(rna_1, rna_2)
Th17_atac <- cbind(atac_1, atac_2)

D2_Treg_1 <- Treg[,Treg$orig.ident == 'Multiome_D2_Treg_1']
D2_Treg_2 <- Treg[,Treg$orig.ident == 'Multiome_D2_Treg_2']
rna_1 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Treg_1/')[["Gene Expression"]][, substr(colnames(D2_Treg_1), 30, 47)]
rna_2 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Treg_2/')[["Gene Expression"]][, substr(colnames(D2_Treg_2), 30, 47)]
atac_1 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Treg_1.rds')[, substr(colnames(D2_Treg_1), 30, 47)]
atac_2 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Treg_2.rds')[, substr(colnames(D2_Treg_2), 30, 47)]
colnames(rna_1) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(D2_Treg_1), 30, 46),
                          'D2_Treg_1')
colnames(atac_1) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(D2_Treg_1), 30, 46),
                           'D2_Treg_1')
colnames(rna_2) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(D2_Treg_2), 30, 46),
                          'D2_Treg_2')
colnames(atac_2) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(D2_Treg_2), 30, 46),
                           'D2_Treg_2')
Treg_rna <- cbind(rna_1, rna_2)
Treg_atac <- cbind(atac_1, atac_2)

D2_Tfh_1 <- Tfh[,Tfh$orig.ident == 'Multiome_D2_Tfh_1']
D2_Tfh_2 <- Tfh[,Tfh$orig.ident == 'Multiome_D2_Tfh_2']
D2_Tfh_3 <- Tfh[,Tfh$orig.ident == 'Multiome_D2_Tfh_3']
rna_1 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Tfh_1/')[["Gene Expression"]][, substr(colnames(D2_Tfh_1), 28, 45)]
rna_2 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Tfh_2/')[["Gene Expression"]][, substr(colnames(D2_Tfh_2), 28, 45)]
rna_3 <- Read10X('Oguchi_Science_2024/raw/E-GEAD-743.matrix/D2_Tfh_3/')[["Gene Expression"]][, substr(colnames(D2_Tfh_3), 28, 45)]
atac_1 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Tfh_1.rds')[, substr(colnames(D2_Tfh_1), 28, 45)]
atac_2 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Tfh_2.rds')[, substr(colnames(D2_Tfh_2), 28, 45)]
atac_3 <- readRDS('z_others/2_cpeak_matrix/data/Oguchi_Science_2024-D2_Tfh_3.rds')[, substr(colnames(D2_Tfh_3), 28, 45)]
colnames(rna_1) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(D2_Tfh_1), 28, 44),
                          'D2_Tfh_1')
colnames(atac_1) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(D2_Tfh_1), 28, 44),
                           'D2_Tfh_1')
colnames(rna_2) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(D2_Tfh_2), 28, 44),
                          'D2_Tfh_2')
colnames(atac_2) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(D2_Tfh_2), 28, 44),
                           'D2_Tfh_2')
colnames(rna_3) <- paste0('Oguchi_Science_2024-',
                          substr(colnames(D2_Tfh_3), 28, 44),
                          'D2_Tfh_3')
colnames(atac_3) <- paste0('Oguchi_Science_2024-',
                           substr(colnames(D2_Tfh_3), 28, 44),
                           'D2_Tfh_3')

Tfh_rna <- cbind(rna_1, rna_2, rna_3)
Tfh_atac <- cbind(atac_1, atac_2, atac_3)


# CD4bluk  59841
# Th1or2  27696
# Activated_CD4_48h  2559
# LAG3  5279
# Th17  11715
# Treg  14691
# Tfh  23704
# total 59841+27696+2559+5279+11715+14691+23704=145,485

rna <- cbind(CD4bluk_rna,Tfh_rna,Th17_rna,Th1or2_rna,Treg_rna,LAG3_rna, Activated_CD4_48h_rna)
atac <- cbind(CD4bluk_atac,Tfh_atac,Th17_atac,Th1or2_atac,Treg_atac,LAG3_atac, Activated_CD4_48h_atac)

saveRDS(rna, 'z_others/3_rna_atac_cell/data/Oguchi_Science_2024-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Oguchi_Science_2024-atac.rds')

In [ ]:
# Swanson_Elife_2021_1
file <- list.files('Swanson_Elife_2021_1/cpeak_matrix/sample/')

rna <- readRDS(paste0('Swanson_Elife_2021_1/processed/', file[1]))@assays[["RNA"]]@counts
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/Swanson_Elife_2021_1-', file[1]))
atac <- atac[,colnames(rna)]
colnames(rna) <- paste0('Swanson_Elife_2021_1-',
                        colnames(rna),'-',
                        substr(file[1], 1, 7))
colnames(atac) <- paste0('Swanson_Elife_2021_1-',
                         colnames(atac),'-',
                         substr(file[1], 1, 7))
for (i in 2:4) {
  rnaa <- readRDS(paste0('Swanson_Elife_2021_1/processed/', file[i]))@assays[["RNA"]]@counts
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/Swanson_Elife_2021_1-', file[i]))
  ataca <- ataca[,colnames(rnaa)]
  colnames(rnaa) <- paste0('Swanson_Elife_2021_1-',
                           colnames(rnaa),'-',
                           substr(file[i], 1, 7))
  colnames(ataca) <- paste0('Swanson_Elife_2021_1-',
                            colnames(ataca),'-',
                            substr(file[i], 1, 7))
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}
saveRDS(rna, 'z_others/3_rna_atac_cell/data/Swanson_Elife_2021_1-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Swanson_Elife_2021_1-atac.rds')

In [ ]:
# Swanson_Elife_2021_2
file <- list.files('Swanson_Elife_2021_2/cpeak_matrix/sample/')

rna_ <- readRDS(paste0('Swanson_Elife_2021_2/processed/', file[1]))
rna <- rna_@assays[["RNA"]]@layers[["counts"]]
colnames(rna) <- colnames(rna_)
rownames(rna) <- rownames(rna_)
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/Swanson_Elife_2021_2-', file[1]))
atac <- atac[,colnames(rna)]
colnames(rna) <- paste0('Swanson_Elife_2021_2-',
                        colnames(rna),'-',
                        substr(file[1], 1, 7))
colnames(atac) <- paste0('Swanson_Elife_2021_2-',
                         colnames(atac),'-',
                         substr(file[1], 1, 7))

rnaa_ <- readRDS(paste0('Swanson_Elife_2021_2/processed/', file[2]))
rnaa <- rnaa_@assays[["RNA"]]@layers[["counts"]]
colnames(rnaa) <- colnames(rnaa_)
rownames(rnaa) <- rownames(rnaa_)
ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/Swanson_Elife_2021_2-', file[2]))
ataca <- ataca[,colnames(rnaa)]
colnames(rnaa) <- paste0('Swanson_Elife_2021_2-',
                         colnames(rnaa),'-',
                         substr(file[2], 1, 7))
colnames(ataca) <- paste0('Swanson_Elife_2021_2-',
                          colnames(ataca),'-',
                          substr(file[2], 1, 7))
rna <- cbind(rna, rnaa)
atac <- cbind(atac, ataca)

saveRDS(rna, 'z_others/3_rna_atac_cell/data/Swanson_Elife_2021_2-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Swanson_Elife_2021_2-atac.rds')

In [ ]:
# Thomson_Nat_Immunol_2024
file <- list.files('Thomson_Nat_Immunol_2024/processed/')[1:19]
ata <- substr(file, 12, 30)
ata <- lapply(ata, function(x) gsub("-", "_", x))

rna <- readRDS(paste0('Thomson_Nat_Immunol_2024/processed/', file[1], '/', file[1], '.rds'))@assays[["RNA"]]@counts
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/Thomson_Nat_Immunol_2024-', ata[[1]], '.rds'))
atac <- atac[,colnames(rna)]
colnames(rna) <- paste0('Thomson_Nat_Immunol_2024-',
                        colnames(rna),'-',
                        ata[[1]])
colnames(atac) <- paste0('Thomson_Nat_Immunol_2024-',
                         colnames(atac),'-',
                         ata[[1]])
for (i in 2:19) {
  rnaa <- readRDS(paste0('Thomson_Nat_Immunol_2024/processed/', file[i], '/', file[i], '.rds'))@assays[["RNA"]]@counts
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/Thomson_Nat_Immunol_2024-', ata[[i]], '.rds'))
  ataca <- ataca[,colnames(rnaa)]
  colnames(rnaa) <- paste0('Thomson_Nat_Immunol_2024-',
                           colnames(rnaa),'-',
                           ata[[i]])
  colnames(ataca) <- paste0('Thomson_Nat_Immunol_2024-',
                            colnames(ataca),'-',
                            ata[[i]])
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}
saveRDS(rna, 'z_others/3_rna_atac_cell/data/Thomson_Nat_Immunol_2024-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Thomson_Nat_Immunol_2024-atac.rds')

In [ ]:
# Wei_Immuni_2023
files <- list.files('Wei_Immuni_2023/processed/')
rna <- readRDS(paste0('Wei_Immuni_2023/processed/', files[1], '/', files[1],'.rds'))@assays[["RNA"]]@counts
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/Wei_Immuni_2023-', files[1],'.rds'))
atac <- atac[,colnames(rna)]
colnames(rna) <- paste0('Wei_Immuni_2023-',
                        substr(colnames(rna), 1, 17),
                        files[1])
colnames(atac) <- paste0('Wei_Immuni_2023-',
                         substr(colnames(atac), 1, 17),
                         files[1])

for (i in 2:10){
  rnaa <- readRDS(paste0('Wei_Immuni_2023/processed/', files[i], '/', files[i],'.rds'))@assays[["RNA"]]@counts
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/Wei_Immuni_2023-', files[i],'.rds'))
  ataca <- ataca[,colnames(rnaa)]
  colnames(rnaa) <- paste0('Wei_Immuni_2023-',
                           substr(colnames(rnaa), 1, 17),
                           files[i])
  colnames(ataca) <- paste0('Wei_Immuni_2023-',
                            substr(colnames(ataca), 1, 17),
                            files[i])
  
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}

saveRDS(rna, 'z_others/3_rna_atac_cell/data/Wei_Immuni_2023-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Wei_Immuni_2023-atac.rds')

In [ ]:
# Weng_Nature_2024
a1_rna <- GSE219014_Aged1@assays$RNA@counts
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_1.rds')
a1_atac <- atac[,colnames(a1_rna)]
colnames(a1_rna) <- paste0('Weng_Nature_2024_1-', colnames(a1_rna))
colnames(a1_atac) <- paste0('Weng_Nature_2024_1-', colnames(a1_atac))

a2_rna <- GSE219057_Aged2@assays$RNA@counts
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_2.rds')
a2_atac <- atac[,colnames(a2_rna)]
colnames(a2_rna) <- paste0('Weng_Nature_2024_2-', colnames(a2_rna))
colnames(a2_atac) <- paste0('Weng_Nature_2024_2-', colnames(a2_atac))

a3_rna <- GSE219106_Young1@assays$RNA@counts
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_3.rds')
a3_atac <- atac[,colnames(a3_rna)]
colnames(a3_rna) <- paste0('Weng_Nature_2024_3-', colnames(a3_rna))
colnames(a3_atac) <- paste0('Weng_Nature_2024_3-', colnames(a3_atac))

a4_rna <- GSE219167_Young1_T2@assays$RNA@counts
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_4.rds')
a4_atac <- atac[,colnames(a4_rna)]
colnames(a4_rna) <- paste0('Weng_Nature_2024_4-', colnames(a4_rna))
colnames(a4_atac) <- paste0('Weng_Nature_2024_4-', colnames(a4_atac))

a5_rna <- GSE219248_Young2@assays$RNA@counts
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_5.rds')
a5_atac <- atac[,colnames(a5_rna)]
colnames(a5_rna) <- paste0('Weng_Nature_2024_5-', colnames(a5_rna))
colnames(a5_atac) <- paste0('Weng_Nature_2024_5-', colnames(a5_atac))

# 6 Corresponding to two samples
a61_rna <- Read10X_h5('Weng_Nature_2024_1/raw/GSE261078_Extended_donors/GSM8133911_Extended_Young_BMMC.filtered_feature_bc_matrix.h5')[["Gene Expression"]]
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_6-Extended_Young_BMMC.rds')
a61_atac <- atac[,colnames(a61_rna)]
colnames(a61_rna) <- paste0('Weng_Nature_2024_6-', substr(colnames(a61_rna), 1, 17), 'BMMC')
colnames(a61_atac) <- paste0('Weng_Nature_2024_6-', substr(colnames(a61_atac), 1, 17), 'BMMC')

a62_rna <- Read10X_h5('Weng_Nature_2024_1/raw/GSE261078_Extended_donors/GSM8133914_Extended_Young_HSPC.filtered_feature_bc_matrix.h5')
a62_rna <- a62_rna[["Gene Expression"]]
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_6-Extended_Young_HSPC.rds')
a62_atac <- atac[,colnames(a62_rna)]
colnames(a62_rna) <- paste0('Weng_Nature_2024_6-', substr(colnames(a62_rna), 1, 17), 'HSPC')
colnames(a62_atac) <- paste0('Weng_Nature_2024_6-', substr(colnames(a62_atac), 1, 17), 'HSPC')


a7_rna <- Read10X_h5('Weng_Nature_2024_1/raw/GSE261078_Extended_donors/GSM8133917_Extended_Aged_BMMC_HSPC.filtered_feature_bc_matrix.h5')[["Gene Expression"]]
atac <- readRDS('z_others/2_cpeak_matrix/data/Weng_Nature_2024_7.rds')
a7_atac <- atac[,colnames(a7_rna)]
colnames(a7_rna) <- paste0('Weng_Nature_2024_7-', colnames(a7_rna))
colnames(a7_atac) <- paste0('Weng_Nature_2024_7-', colnames(a7_atac))

rna <- cbind(a1_rna,
             a2_rna,
             a3_rna,
             a4_rna,
             a5_rna,
             a61_rna,
             a62_rna,
             a7_rna)
atac <- cbind(a1_atac,
             a2_atac,
             a3_atac,
             a4_atac,
             a5_atac,
             a61_atac,
             a62_atac,
             a7_atac)

saveRDS(rna, 'z_others/3_rna_atac_cell/data/Weng_Nature_2024-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Weng_Nature_2024-atac.rds')

In [ ]:
# Wimmers_Cell_2023
file <- list.files('Wimmers_Cell_2023/processed/')
rna <- Read10X(paste0('Wimmers_Cell_2023/raw/GSE239799/', file[1]))[["Gene Expression"]]
atac <- readRDS(paste0('z_others/2_cpeak_matrix/data/Wimmers_Cell_2023-',substr(file[1], 12, 30),'.rds'))
atac <- atac[, colnames(rna)]
colnames(rna) <- paste0('Wimmers_Cell_2023-',
                        substr(colnames(rna),1,17),
                        substr(file[1], 12, 30))
colnames(atac) <- paste0('Wimmers_Cell_2023-',
                        substr(colnames(atac),1,17),
                        substr(file[1], 12, 30))

for (i in 2:length(file)){
  rnaa <- Read10X(paste0('Wimmers_Cell_2023/raw/GSE239799/', file[i]))[["Gene Expression"]]
  ataca <- readRDS(paste0('z_others/2_cpeak_matrix/data/Wimmers_Cell_2023-',substr(file[i], 12, 30),'.rds'))
  ataca <- ataca[, colnames(rnaa)]
  colnames(rnaa) <- paste0('Wimmers_Cell_2023-',
                          substr(colnames(rnaa),1,17),
                          substr(file[i], 12, 30))
  colnames(ataca) <- paste0('Wimmers_Cell_2023-',
                           substr(colnames(ataca),1,17),
                           substr(file[i], 12, 30))
  
  rna <- cbind(rna, rnaa)
  atac <- cbind(atac, ataca)
}

meta <- read.csv('Wimmers_Cell_2023/raw/wimmers2023.celltype.csv.gz')
meta$barcode <- sapply(meta$cell, function(x) substr(x, nchar(x) - 17, nchar(x) - 1))
meta$id <- paste0('Wimmers_Cell_2023-',
                  meta$barcode,
                  substr(meta$cell, 1, 9))
rna <- rna[, meta$id]
atac <- atac[, meta$id]

saveRDS(rna, 'z_others/3_rna_atac_cell/data/Wimmers_Cell_2023-rna.rds')
saveRDS(atac, 'z_others/3_rna_atac_cell/data/Wimmers_Cell_2023-atac.rds')

# Merge matrix

In [ ]:
setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/')

list <- list.files('3_rna_atac_cell/data/', pattern = 'atac.rds')
dat <- readRDS(paste0('3_rna_atac_cell/data/', list[1]))
for (i in 17:17){
  data <- readRDS(paste0('3_rna_atac_cell/data/', list[17]))
  dat <- cbind(dat, data)
}

writeMM(dat, '3_rna_atac_cell/data_rna/rna_16_17.mtx')

cell <- as.data.frame(dat@Dimnames[[2]])
colnames(cell) <- 'cell_id'
write.csv(cell, '3_rna_atac_cell/data_rna/cell_16_17.csv')

peak <- as.data.frame(dat@Dimnames[[1]])
colnames(peak) <- 'peak'
write.csv(peak, '3_rna_atac_cell/data_atac/peak.csv')

# Handle separately
# 6-42,319 gene
dat <- readRDS(paste0('3_rna_atac_cell/data/', list[6]))
length(intersect(dat@Dimnames[[1]], data@Dimnames[[1]]))
# intersection-30374
dat <- dat[intersect(dat@Dimnames[[1]], data@Dimnames[[1]]),]

other_gene <- setdiff(data@Dimnames[[1]], dat@Dimnames[[1]])
mat <- Matrix(0, nrow = 6227, ncol = 874480, sparse = TRUE)
mat@Dimnames[[1]] <- other_gene
mat@Dimnames[[2]] <- dat@Dimnames[[2]]

matrix <- rbind(dat, mat)
matrix <- matrix[data@Dimnames[[1]],]
writeMM(matrix, '3_rna_atac_cell/data_rna/rna_6.mtx')

cell <- as.data.frame(matrix@Dimnames[[2]])
colnames(cell) <- 'cell_id'
write.csv(cell, '3_rna_atac_cell/data_rna/cell_6.csv')